# 5) Flashcard Drill Agent

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Provide flashcard drilling with self-rating; show 'hard' cards more often over time.

## Simple rules

- Show one question.
- Reveal answer.
- Rate easy/medium/hard.
- Weight selection toward hard cards.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "05_flashcards_memory.json"

FLASHCARDS = [
    {"q": "Define recursion in one line.", "a": "A function calling itself on smaller subproblems until base case."},
    {"q": "What is a stack?", "a": "A LIFO data structure supporting push/pop."},
    {"q": "What is a Python dict?", "a": "A key-value mapping structure (hash table-like)."},
    {"q": "What is a primary key?", "a": "A column (or set) that uniquely identifies each row."},
]

def weighted_card_index(memory: Dict[str, Any]) -> int:
    ratings = memory.get("ratings", {})
    weights = []
    for c in FLASHCARDS:
        key = c["q"]
        r = ratings.get(key, {"easy": 0, "medium": 0, "hard": 0})
        w = (int(r["hard"]) + 2) + (int(r["medium"]) + 1) + int(r["easy"])
        weights.append(w)
    return random.choices(range(len(FLASHCARDS)), weights=weights, k=1)[0]

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}
    memory.setdefault("ratings", {})

    Tools.notify("Flashcard Drill Agent is running.")
    Tools.notify("Press Enter to draw a card. Type 'stop' to end.")

    cmd = Tools.ask("Start?")
    while True:
        if should_stop(cmd):
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Saved. Bye!")
            break

        idx = weighted_card_index(memory)
        card = FLASHCARDS[idx]
        Tools.notify(f"Q: {card['q']}")
        reveal = Tools.ask("Press Enter to reveal (or stop):")
        if should_stop(reveal):
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Saved. Bye!")
            break
        Tools.notify(f"A: {card['a']}")

        rating = Tools.ask("Rate: easy / medium / hard")
        rating = rating.strip().lower()
        if rating not in {"easy", "medium", "hard"}:
            rating = "medium"

        memory["ratings"].setdefault(card["q"], {"easy": 0, "medium": 0, "hard": 0})
        memory["ratings"][card["q"]][rating] += 1

        env.execute(Action("save_memory", {}), memory)
        cmd = Tools.ask("Next? (Enter / stop)")

run_agent()


## Easy extensions

- Let students add cards via input.
- Add topics and filtering.
- Simple spaced repetition.